# 04 — Deploy an Agent to Azure Container Apps

This notebook walks you through building a Docker image, pushing it to Azure Container
Registry (ACR), and deploying your agent to Azure Container Apps (ACA) using the
included Bicep templates.

It's a natural follow-up to **03_scaffold_agent.ipynb** — once you've scaffolded and
tested an agent locally, this notebook takes it to production.

> **Full reference**: See the [Deployment Guide](../docs/deployment-guide.md) for
> all options including environment sizing, Key Vault secrets, managed identity,
> RBAC, VNET integration, and troubleshooting.

## Prerequisites

Before starting, make sure you have:

- **Azure CLI** >= 2.60 with the Bicep extension — [Install](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli)
- **Docker** installed (or use ACR Tasks to build remotely)
- An **Azure subscription** with Contributor + User Access Administrator roles
- An **Azure AI Foundry project** with a model deployment (e.g., `gpt-4o`)
- Authenticated session: `az login`

## Configuration

Set your deployment variables below. Update these to match your Azure environment.

In [ ]:
import sys, pathlib

# Ensure the local project root is first on sys.path
_project_root = str(pathlib.Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# ----- Deployment Configuration -----

# Agent to deploy (must be a registered agent)
AGENT_NAME = "code-helper"

# Azure resource names
RESOURCE_GROUP = "agentframework-rg"
LOCATION = "eastus"
ACR_NAME = "<your-acr-name>"                # Must be globally unique
ACR_SUFFIX = "azurecr.io"                 # Use "azurecr.us" for Azure Government
IMAGE_TAG = "v1"

# Azure AI Foundry endpoint
FOUNDRY_ENDPOINT = "https://<your-project>.services.ai.azure.com/api/projects/<your-project>"

# Sovereign / Government cloud authority host (leave empty for commercial cloud)
# Use "https://login.microsoftonline.us" for Azure Government
AZURE_AUTHORITY_HOST = ""

# Foundry resource group (for RBAC in Step 9)
# Set this if your Foundry/AI Services resource is in a different resource group
FOUNDRY_RESOURCE_GROUP = ""                 # e.g., "my-foundry-rg" (leave empty to search all)

# Environment (dev, qa, or prod)
ENVIRONMENT = "dev"

# ----- Knowledge Base / Search Grounding (optional) -----
# Leave empty ("") to deploy without search grounding.
# See docs/knowledge-search-guide.md for details.

# Option A: Single index
AZURE_AI_SEARCH_ENDPOINT = ""              # e.g., "https://<search-service>.search.windows.net"
AZURE_AI_SEARCH_INDEX_NAME = ""            # e.g., "my-index"
AZURE_AI_SEARCH_SEMANTIC_CONFIG = ""       # e.g., "my-semantic-config" (optional)

# Option C: Multiple indexes (JSON array string)
AZURE_AI_SEARCH_INDEXES = ""               # e.g., '[{"endpoint":"https://...","index_name":"products"}]'

In [ ]:
# Derived variables (no changes needed)
IMAGE_REF = f"{ACR_NAME}.{ACR_SUFFIX}/agentframework:{IMAGE_TAG}"

print("Deployment plan:")
print(f"  Agent:          {AGENT_NAME}")
print(f"  Resource group: {RESOURCE_GROUP}")
print(f"  Location:       {LOCATION}")
print(f"  ACR:            {ACR_NAME}")
print(f"  Image:          {IMAGE_REF}")
print(f"  Environment:    {ENVIRONMENT}")
print(f"  Foundry:        {FOUNDRY_ENDPOINT[:60]}…")
if AZURE_AI_SEARCH_ENDPOINT:
    print(f"  Search:         {AZURE_AI_SEARCH_ENDPOINT} / {AZURE_AI_SEARCH_INDEX_NAME}")
if AZURE_AI_SEARCH_INDEXES:
    print(f"  Multi-index:    {AZURE_AI_SEARCH_INDEXES[:80]}…")
if not AZURE_AI_SEARCH_ENDPOINT and not AZURE_AI_SEARCH_INDEXES:
    print(f"  Search:         (not configured — model-only answers)")

## Step 1: Verify Azure Login

Confirm you're authenticated and using the correct subscription.

In [ ]:
import subprocess, json, shlex

def az(cmd: str, *, timeout: int = 300, stream: bool = False) -> subprocess.CompletedProcess:
    """Run an Azure CLI command safely (no shell hang).
    
    Args:
        cmd: The az CLI args (without leading 'az'), e.g. "group create --name rg1 ..."
        timeout: Max seconds to wait (default 5 min; use longer for builds/deploys).
        stream: If True, print stdout/stderr live line-by-line.
    Returns:
        CompletedProcess with .stdout, .stderr, .returncode
    """
    args = ["az"] + shlex.split(cmd)
    if stream:
        proc = subprocess.Popen(
            args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        lines = []
        for raw_line in proc.stdout:
            line = raw_line.decode("utf-8", errors="replace")
            print(line, end="")
            lines.append(line)
        proc.wait(timeout=timeout)
        return subprocess.CompletedProcess(args, proc.returncode, stdout="".join(lines), stderr="")
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout)

# Quick login check
result = az("account show -o json", timeout=10)
if result.returncode == 0:
    acct = json.loads(result.stdout)
    print(f"✓ Logged in as: {acct['name']} ({acct['id'][:8]}…)")
else:
    print("⚠️  Not logged in. Run in a terminal:\n    az login --use-device-code")
    print(f"\n{result.stderr.strip()}")

In [ ]:
# Ensure Bicep CLI is up-to-date (required for modern API versions and syntax)
r = az("bicep upgrade", timeout=60)
if r.returncode == 0:
    print("✓ Bicep CLI upgraded")
else:
    # Fresh install if upgrade fails (no prior install)
    r = az("bicep install", timeout=60)
    print("✓ Bicep CLI installed" if r.returncode == 0 else f"⚠️  {r.stderr.strip()}")

r = az("bicep version", timeout=10)
print(f"  {r.stdout.strip()}" if r.returncode == 0 else r.stderr.strip())

## Step 2: Create Resource Group and ACR

Skip this step if you already have a resource group and ACR.

In [ ]:
# Create resource group
r = az(f"group create --name {RESOURCE_GROUP} --location {LOCATION} -o table")
print(r.stdout or r.stderr)

In [ ]:
# Create Azure Container Registry
r = az(f"acr create --name {ACR_NAME} --resource-group {RESOURCE_GROUP} --sku Basic --admin-enabled false -o table")
print(r.stdout or r.stderr)

## Step 3: Build and Push Docker Image

You have two options:

- **ACR Tasks** (remote build — no local Docker needed)
- **Local build** then push

Choose one approach below.

In [ ]:
# Option A: Build remotely with ACR Tasks (recommended — no local Docker required)
# Streams output live since builds take a while
import os
os.chdir(_project_root)
r = az(f"acr build --registry {ACR_NAME} --image agentframework:{IMAGE_TAG} .", timeout=600, stream=True)
if r.returncode != 0:
    print(f"\n⚠️  Build failed (exit {r.returncode})")

In [ ]:
# Option B: Build locally and push (uncomment to use)
# import os; os.chdir(_project_root)
# !docker build -t {IMAGE_REF} .
# r = az(f"acr login --name {ACR_NAME}", timeout=30); print(r.stdout or r.stderr)
# !docker push {IMAGE_REF}

## Step 4: Update Bicep Parameters

Before deploying, ensure `infra/parameters.dev.bicepparam` has your ACR resource ID.
The cell below reads the current value so you can verify it.

In [ ]:
params_file = "infra/parameters.dev.bicepparam"
with open(params_file) as f:
    content = f.read()

# Extract the acrResourceId line
for line in content.splitlines():
    if "acrResourceId" in line and not line.strip().startswith("//"):
        print(f"Current setting:\n  {line.strip()}")
        break

if "<subscription-id>" in content or "<acr-rg>" in content or "<acr-name>" in content:
    print("\n⚠️  Placeholder values detected! Update infra/parameters.dev.bicepparam with your actual ACR resource ID:")
    print(f"  /subscriptions/<sub-id>/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.ContainerRegistry/registries/{ACR_NAME}")
else:
    print("\n✓ ACR resource ID appears to be configured")

## Step 5: Preview Deployment (What-If)

Run a what-if to see what Azure resources will be created or modified,
without actually deploying anything.

In [ ]:
print(f"Running what-if for agent '{AGENT_NAME}' in '{ENVIRONMENT}'…")

In [ ]:
env_vars = [
    {"name": "AGENT_NAME", "value": AGENT_NAME},
    {"name": "AZURE_AI_PROJECT_ENDPOINT", "value": FOUNDRY_ENDPOINT},
    {"name": "ENVIRONMENT", "value": ENVIRONMENT},
]

# Sovereign / Government cloud support
if AZURE_AUTHORITY_HOST:
    env_vars.append({"name": "AZURE_AUTHORITY_HOST", "value": AZURE_AUTHORITY_HOST})

# Conditionally add search grounding vars
if AZURE_AI_SEARCH_ENDPOINT:
    env_vars.append({"name": "AZURE_AI_SEARCH_ENDPOINT", "value": AZURE_AI_SEARCH_ENDPOINT})
if AZURE_AI_SEARCH_INDEX_NAME:
    env_vars.append({"name": "AZURE_AI_SEARCH_INDEX_NAME", "value": AZURE_AI_SEARCH_INDEX_NAME})
if AZURE_AI_SEARCH_SEMANTIC_CONFIG:
    env_vars.append({"name": "AZURE_AI_SEARCH_SEMANTIC_CONFIG", "value": AZURE_AI_SEARCH_SEMANTIC_CONFIG})
if AZURE_AI_SEARCH_INDEXES:
    env_vars.append({"name": "AZURE_AI_SEARCH_INDEXES", "value": AZURE_AI_SEARCH_INDEXES})

env_vars_json = json.dumps(env_vars)

# Read the bicepparam to extract static values (avoids supplemental-parameter Bicep CLI limitation)
import re

params_file = "infra/parameters.dev.bicepparam"
with open(params_file) as f:
    params_content = f.read()

def extract_param(name: str, content: str) -> str:
    m = re.search(rf"param\s+{name}\s*=\s*'([^']*)'", content)
    return m.group(1) if m else ""

acr_resource_id = extract_param("acrResourceId", params_content)

r = az(
    f"deployment group what-if"
    f" --resource-group {RESOURCE_GROUP}"
    f" --template-file infra/main.bicep"
    f" --parameters environmentName={ENVIRONMENT}"
    f" --parameters location={LOCATION}"
    f" --parameters appName={AGENT_NAME}"
    f" --parameters containerImage={IMAGE_REF}"
    f" --parameters acrResourceId={shlex.quote(acr_resource_id)}"
    f" --parameters appEnvVars={shlex.quote(env_vars_json)}",
    timeout=120,
)
print(r.stdout or r.stderr)

## Step 6: Deploy to ACA

This creates all required resources via Bicep:
- **User-assigned managed identity** (`id-{agent}-{env}`)
- **ACR role assignment** (AcrPull)
- **Log Analytics workspace** (`law-{agent}-{env}`)
- **ACA managed environment** (`env-{agent}-{env}`)
- **Container app** (`ca-{agent}-{env}`)

Re-running is idempotent — existing resources are updated in place.

In [ ]:
import datetime

deployment_name = f"agent-{AGENT_NAME}-{ENVIRONMENT}-{datetime.datetime.now():%Y%m%d%H%M}"
print(f"Deployment name: {deployment_name}\n")

In [ ]:
env_vars = [
    {"name": "AGENT_NAME", "value": AGENT_NAME},
    {"name": "AZURE_AI_PROJECT_ENDPOINT", "value": FOUNDRY_ENDPOINT},
    {"name": "ENVIRONMENT", "value": ENVIRONMENT},
]

# Sovereign / Government cloud support
if AZURE_AUTHORITY_HOST:
    env_vars.append({"name": "AZURE_AUTHORITY_HOST", "value": AZURE_AUTHORITY_HOST})

# Conditionally add search grounding vars
if AZURE_AI_SEARCH_ENDPOINT:
    env_vars.append({"name": "AZURE_AI_SEARCH_ENDPOINT", "value": AZURE_AI_SEARCH_ENDPOINT})
if AZURE_AI_SEARCH_INDEX_NAME:
    env_vars.append({"name": "AZURE_AI_SEARCH_INDEX_NAME", "value": AZURE_AI_SEARCH_INDEX_NAME})
if AZURE_AI_SEARCH_SEMANTIC_CONFIG:
    env_vars.append({"name": "AZURE_AI_SEARCH_SEMANTIC_CONFIG", "value": AZURE_AI_SEARCH_SEMANTIC_CONFIG})
if AZURE_AI_SEARCH_INDEXES:
    env_vars.append({"name": "AZURE_AI_SEARCH_INDEXES", "value": AZURE_AI_SEARCH_INDEXES})

env_vars_json = json.dumps(env_vars)

# Read the bicepparam to extract static values (avoids supplemental-parameter Bicep CLI limitation)
import re

params_file = "infra/parameters.dev.bicepparam"
with open(params_file) as f:
    params_content = f.read()

def extract_param(name: str, content: str) -> str:
    m = re.search(rf"param\s+{name}\s*=\s*'([^']*)'", content)
    return m.group(1) if m else ""

acr_resource_id = extract_param("acrResourceId", params_content)

r = az(
    f"deployment group create"
    f" --resource-group {RESOURCE_GROUP}"
    f" --template-file infra/main.bicep"
    f" --parameters environmentName={ENVIRONMENT}"
    f" --parameters location={LOCATION}"
    f" --parameters appName={AGENT_NAME}"
    f" --parameters containerImage={IMAGE_REF}"
    f" --parameters acrResourceId={shlex.quote(acr_resource_id)}"
    f" --parameters appEnvVars={shlex.quote(env_vars_json)}"
    f" --name {deployment_name}"
    f" --mode Incremental",
    timeout=600,
    stream=True,
)
if r.returncode != 0:
    print(f"\n⚠️  Deployment failed (exit {r.returncode})")
else:
    print(f"\n✓ Deployment '{deployment_name}' completed")

## Step 7: Verify Deployment

Get the container app's FQDN and test the `/responses` endpoint.

In [ ]:
# Get the FQDN of the deployed container app
r = az(
    f"containerapp show"
    f" --name ca-{AGENT_NAME}-{ENVIRONMENT}"
    f" --resource-group {RESOURCE_GROUP}"
    f' --query "properties.configuration.ingress.fqdn" -o tsv',
    timeout=30,
)

FQDN = r.stdout.strip()
if FQDN:
    print(f"✓ Container app is live!")
    print(f"  FQDN:     {FQDN}")
    print(f"  Endpoint: https://{FQDN}/responses")
else:
    print("⚠️  Could not retrieve FQDN — check deployment logs")
    print(r.stderr.strip())

In [ ]:
# Send a test request to the deployed agent
import urllib.request, urllib.error

if FQDN:
    url = f"https://{FQDN}/responses"
    payload = json.dumps({"input": "Hello! Say OK if you are running."}).encode()
    req = urllib.request.Request(url, data=payload, headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            print(resp.read().decode())
    except urllib.error.HTTPError as e:
        print(f"⚠️  HTTP {e.code}: {e.read().decode()}")
    except urllib.error.URLError as e:
        print(f"⚠️  Connection error: {e.reason}")
else:
    print("⚠️  No FQDN — run the previous cell first")

## Step 8: View Container Logs

Check the container logs to verify the agent started correctly.

In [ ]:
r = az(
    f"containerapp logs show"
    f" --name ca-{AGENT_NAME}-{ENVIRONMENT}"
    f" --resource-group {RESOURCE_GROUP}"
    f" --tail 30",
    timeout=30,
)
print(r.stdout or r.stderr)

## Step 9: Assign Foundry RBAC (If Using Managed Identity)

For production, use managed identity instead of service principal secrets.
The managed identity needs **Cognitive Services User** on your Foundry resource.

In [ ]:
# Get the managed identity's principal ID
r = az(
    f"identity show"
    f" --name id-{AGENT_NAME}-{ENVIRONMENT}"
    f" --resource-group {RESOURCE_GROUP}"
    f" --query principalId -o tsv",
    timeout=30,
)

PRINCIPAL_ID = r.stdout.strip()
if not PRINCIPAL_ID:
    print("⚠️  Could not retrieve principal ID")
    print(r.stderr.strip())
else:
    print(f"✓ Managed identity principal: {PRINCIPAL_ID}")

    # Find the AI Services account
    rg_flag = f" -g {FOUNDRY_RESOURCE_GROUP}" if FOUNDRY_RESOURCE_GROUP else ""
    r2 = az(f"cognitiveservices account list{rg_flag} -o json", timeout=30)

    foundry_scope = None
    accounts = json.loads(r2.stdout or "[]") if r2.returncode == 0 else []

    # Match by endpoint hostname from FOUNDRY_ENDPOINT
    import re as _re
    _m = _re.match(r"https://([^.]+)\.", FOUNDRY_ENDPOINT)
    foundry_host = _m.group(1) if _m else None

    for acct in accounts:
        endpoint = acct.get("properties", {}).get("endpoint", "")
        if foundry_host and foundry_host in endpoint:
            foundry_scope = acct["id"]
            break
        if acct["name"] == foundry_host:
            foundry_scope = acct["id"]
            break

    if foundry_scope:
        print(f"  Foundry resource: {foundry_scope.split('/')[-1]}")

        # Assign "Cognitive Services User" — required for AIServices/agents/* data actions.
        # Note: "Azure AI Developer" only covers OpenAI/* and is NOT sufficient.
        ROLE = "Cognitive Services User"
        print(f"  Assigning '{ROLE}' role…")
        r4 = az(
            f'role assignment create'
            f' --assignee-object-id "{PRINCIPAL_ID}"'
            f' --assignee-principal-type ServicePrincipal'
            f' --role "{ROLE}"'
            f' --scope "{foundry_scope}"',
            timeout=60,
        )
        if r4.returncode == 0:
            print("  ✓ Role assigned successfully")
            print(f"\n  ⏳ Wait 1-2 min for propagation, then restart:")
            print(f"     az containerapp revision restart -n ca-{AGENT_NAME}-{ENVIRONMENT} -g {RESOURCE_GROUP} --revision <revision>")
        else:
            err = r4.stderr.strip()
            if "already exists" in err.lower() or "conflict" in err.lower():
                print("  ✓ Role already assigned")
            else:
                print(f"  ⚠️  {err}")
    elif accounts:
        print(f"\n⚠️  Could not match endpoint hostname '{foundry_host}'. Available accounts:")
        for acct in accounts:
            print(f"    {acct['name']}  →  {acct['id']}")
        print(f"\n  Set FOUNDRY_RESOURCE_GROUP in the config cell or assign manually:")
        print(f'  az role assignment create --assignee-object-id "{PRINCIPAL_ID}" --assignee-principal-type ServicePrincipal --role "Cognitive Services User" --scope <id-from-above>')
    else:
        print(f"\n⚠️  No AI Services accounts found{' in ' + FOUNDRY_RESOURCE_GROUP if FOUNDRY_RESOURCE_GROUP else ''}.")
        print(f"  → Set FOUNDRY_RESOURCE_GROUP in the Configuration cell to the")
        print(f"    resource group containing your Azure AI Foundry project.")
        print(f"  → If it's in another subscription, switch first:")
        print(f"    az account set --subscription <sub-id>")

## Step 10: Update a Deployment

To push a new version, build a new image tag and re-run the Bicep deployment.

In [ ]:
NEW_TAG = "v2"
NEW_IMAGE = f"{ACR_NAME}.{ACR_SUFFIX}/agentframework:{NEW_TAG}"

print(f"To update the deployment:")
print(f"  1. Build new image:  az acr build --registry {ACR_NAME} --image agentframework:{NEW_TAG} .")
print(f"  2. Re-deploy:        Change IMAGE_TAG to '{NEW_TAG}' above and re-run Steps 6-7")
print(f"\nBicep handles rolling updates — ACA provisions a new revision automatically.")

## Deploy Multiple Agents (Optional)

Each agent runs as a **separate container app** using the same Docker image.
Just change `AGENT_NAME` at the top and re-run Steps 5-7.

```
                    ┌─────────────────────────┐
                    │   Same Docker Image      │
                    │   (agentframework:v1)    │
                    └────┬──────────┬──────────┘
                         │          │
              ┌──────────▼──┐  ┌───▼───────────┐
              │ ca-code-     │  │ ca-doc-        │
              │ helper-dev   │  │ assistant-dev  │
              │ AGENT_NAME=  │  │ AGENT_NAME=    │
              │ code-helper  │  │ doc-assistant   │
              └──────────────┘  └────────────────┘
```

The `AGENT_NAME` env var selects which agent the container serves.

## Troubleshooting

| Issue | Resolution |
|-------|------------|
| `DefaultAzureCredential failed` | Run `az login` (local) or verify `AZURE_CLIENT_ID` is set in the container (Bicep injects it automatically for user-assigned MI) |
| `Connection refused on :8088` | Verify `AGENT_NAME` matches a registered agent |
| `403` / `PermissionDenied` on Foundry | Assign **Cognitive Services User** role (Step 9). Note: **Azure AI Developer** is NOT sufficient — it only covers `OpenAI/*` data actions, not `AIServices/agents/*` |
| Container exits immediately | Check logs: Step 8 above |
| Image pull failure | Verify ACR role assignment completed in Bicep deployment |
| ACA returns 404 page | Check that `ingressExternal` is `true` in main.bicep (default: true) |
| `TargetPort does not match` | Ensure `targetPort` is `8088` in main.bicep (default: 8088) |

See the [Deployment Guide](../docs/deployment-guide.md) for more details.

## Next Steps

- Set up **CI/CD** with the `deploy.yml` GitHub Actions workflow
- Configure **Key Vault secrets** for sensitive env vars — see [Deployment Guide](../docs/deployment-guide.md#environment-configuration)
- Scale to **production** by switching to `parameters.prod.bicepparam`
- Add **VNET integration** for private networking